# Exercises XP – Week 4, Day 5: Introduction to Machine Learning
Based on the course's student notebook template. Dataset for Exercises 2–3: **Loan Prediction** (`loan_prediction.csv`).

## 🌟 Exercise 1: Defining the Problem and Data Collection for Loan Default Prediction
### Instructions
- Write a clear problem statement for predicting loan defaults.
- Identify and list the types of data you would need for this project.
- Discuss the sources where you can collect this data.

### Your answer

**Problem statement.** The goal of this project is to predict whether a loan applicant will **default** (fail to repay their loan as agreed), so that the bank can decide whether to **approve, reject, or adjust** (amount, interest rate, guarantees) each new loan application. The target variable is a binary label, `default` (1 = the borrower defaulted, 0 = the loan was repaid), and the model will output a probability of default for each applicant at the moment they apply.

**Data types we would collect:**
- **Personal and demographic details** (age, marital status, number of dependents, education). These describe the applicant's life situation and financial responsibilities, which affect how much of their income is available to repay a loan.
- **Employment information** (employment status, type of job, self-employed or not, years in current job). Stable, long-term employment usually means a more reliable income, and therefore a lower risk of default.
- **Income and financial situation** (monthly income of the applicant and co-applicant, existing debts, savings, debt-to-income ratio). The more income is left after existing obligations, the easier it is to make the new loan payments.
- **Credit history** (credit score, number of past loans, late payments, previous defaults, number of open credit lines). Past repayment behaviour is usually the strongest single predictor of future repayment behaviour.
- **Loan details** (loan amount, loan term, interest rate, purpose of the loan). Larger loans, longer terms and higher monthly payments relative to income increase the burden on the borrower.
- **Collateral / property information** (whether the loan is secured, property value, property location – urban, semi-urban or rural). Collateral reduces the lender's loss and can change the borrower's incentive to repay.
- **Repayment history on this loan** (the outcome label): whether past loans were repaid on time, late, or defaulted – this is what the model learns to predict.

**Data sources and how to obtain them:**
- **The bank's internal records** (loan application forms, account and transaction history, past loan outcomes). These would be extracted from the bank's databases, joined on a customer ID, and form the core training dataset with the historical default labels.
- **Credit bureaus** (e.g. Experian, Equifax, TransUnion, or in Israel, the Bank of Israel's central credit register). Credit reports and scores would be requested through an official API, with the applicant's consent, at the time of application.
- **Employment and income verification** (pay slips, tax records, employer confirmation). These documents are collected during the application and the key values are entered into the system.
- **Public and third-party data** (property registries for collateral value, macroeconomic data such as unemployment and interest rates). These would be added through official datasets or data providers and joined by date and region.

**Risks and constraints.** Loan data is highly **sensitive personal data**, so the project must follow privacy laws (such as GDPR or local financial regulation), collect only the data that is needed, store it securely, and use it only with the applicant's consent. Credit decisions are **regulated**: the model must not discriminate, directly or indirectly, based on protected characteristics such as gender, ethnicity or religion, and the bank must be able to **explain** why an application was rejected. **Data quality** problems (missing incomes, typing errors, outdated credit reports) can mislead the model, so data must be validated and cleaned. There is also a **sampling bias**: we only know the outcome for loans that were *approved* in the past, so the training data under-represents the people who were rejected, which can make the model over-confident about them. Finally, good **governance** is needed – documenting the data and model, monitoring performance over time (because economic conditions change), and having humans review borderline or high-impact decisions.

## 🌟 Exercise 2: Feature Selection and Model Choice for Loan Default Prediction
### Instructions
From the Loan Prediction dataset, identify which features might be most relevant for predicting loan defaults. Justify your choice of features.

In [ ]:
# If you run this in Google Colab, first upload loan_prediction.csv using the folder icon on the left.
import pandas as pd
import numpy as np

df = pd.read_csv("loan_prediction.csv")
print(df.shape)
df.head()

In [ ]:
# Missing values per column
df.isnull().sum()

In [ ]:
# The target in this dataset is Loan_Status (Y = approved, N = not approved).
# We treat "N" as the risky outcome (our stand-in for default) and look at the rate of N per category.
df["risky"] = (df["Loan_Status"] == "N").astype(int)
print("Overall share of risky (N) applications:", round(df["risky"].mean(), 3))

for col in ["Credit_History", "Property_Area", "Married", "Education", "Dependents", "Self_Employed", "Gender"]:
    print(f"\n{col}:")
    print(df.groupby(col)["risky"].agg(rate="mean", count="count").round(3))

In [ ]:
# Income and loan size: compare medians for approved (Y) vs. not approved (N)
df["TotalIncome"] = df["ApplicantIncome"] + df["CoapplicantIncome"]
df["LoanToIncome"] = df["LoanAmount"] / df["TotalIncome"]
df.groupby("Loan_Status")[["ApplicantIncome", "CoapplicantIncome", "TotalIncome", "LoanAmount", "LoanToIncome"]].median().round(3)

In [ ]:
selected_features = [
    "Credit_History", "TotalIncome", "LoanAmount", "Loan_Amount_Term",
    "LoanToIncome", "Property_Area", "Married", "Education", "Dependents",
]
excluded_features = ["Loan_ID", "Gender", "Self_Employed"]
print("Selected:", selected_features)
print("Excluded:", excluded_features)

### Your justification
**Important note about the target:** in this dataset the target is `Loan_Status` – whether the loan was **approved** (Y) or not (N) – not whether the borrower actually defaulted later. I therefore treat "not approved" as a stand-in for "high risk of default"; a real default model would need repayment outcomes.

**Selected features:** `Credit_History`, `TotalIncome` (ApplicantIncome + CoapplicantIncome), `LoanAmount`, `Loan_Amount_Term`, `LoanToIncome`, `Property_Area`, `Married`, `Education` and `Dependents`.

- **Credit_History** is by far the strongest feature: applicants with a credit history that does **not** meet guidelines (0) were rejected **92%** of the time, compared with **20%** for those with a good history (1). Past repayment behaviour is the best indicator of future repayment.
- **TotalIncome** shows how much money the household has available to repay the loan. Applicant income alone barely differs between approved and rejected groups, but approved applicants have a much higher **co-applicant** income (median ~1,240 vs. ~270), so the combined income is the more useful measure.
- **LoanAmount** and **Loan_Amount_Term** together determine the size of the monthly payment; a larger loan or a shorter term means a heavier repayment burden.
- **LoanToIncome** (a new feature) measures the loan relative to what the applicant earns, which is closer to real affordability than either number on its own.
- **Property_Area** relates to the value and type of the property used as collateral: semi-urban applicants were rejected least often (**23%**) and rural applicants most often (**39%**).
- **Married** shows a clear difference (rejection rate 28% for married vs. 37% for unmarried applicants), probably because married applicants more often have a co-applicant and a second income.
- **Education** matters as an indicator of income stability: non-graduates were rejected more often (**39%** vs. **29%**).
- **Dependents** reflects financial obligations: more dependents means more of the income is already committed to family expenses.

**Excluded features:**
- **Loan_ID** is just an identifier with no relationship to risk, and keeping it could let the model memorise individual rows.
- **Gender** is excluded because using it in credit decisions is discriminatory and usually illegal, and the data shows almost no difference anyway (33% vs. 31%).
- **Self_Employed** is excluded because it shows no difference in this data (31.4% vs. 31.7%), so it adds noise rather than information.

**Encoding and missing values:** I would one-hot encode the nominal categorical features (`Property_Area`, `Married`, `Education`) and treat `Dependents` as an ordinal number by converting "3+" to 3. For missing values, I would impute categorical features with their most frequent value, numerical features like `LoanAmount` with the median (because incomes and loan amounts are skewed), and for `Credit_History` I would add a separate "unknown" indicator, because missing credit information can itself be a risk signal.

## 🌟 Exercise 3: Training, Evaluating, and Optimizing the Model
### Instructions
Which model(s) would you pick for a Loan Prediction? Outline the steps to evaluate the model's performance, mentioning specific metrics that would be relevant to evaluate the model.

In [ ]:
# A small real demonstration of the evaluation plan described below, on this dataset.
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix, classification_report)

data = df.copy()
data["Dependents"] = data["Dependents"].replace("3+", "3").astype(float)
X = data[selected_features]
y = data["risky"]      # 1 = not approved (risky), 0 = approved

numeric = ["Credit_History", "TotalIncome", "LoanAmount", "Loan_Amount_Term", "LoanToIncome", "Dependents"]
categorical = ["Property_Area", "Married", "Education"]

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical),
])

models = {
    "Logistic Regression": LogisticRegression(class_weight="balanced", max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=5, class_weight="balanced", random_state=42),
}

# 1) Stratified train/test split – the test set is kept aside until the very end
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# 2) Stratified 5-fold cross-validation on the training set only
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, model in models.items():
    pipe = Pipeline([("prep", preprocess), ("model", model)])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc")
    print(f"{name}: cross-validated ROC-AUC = {scores.mean():.3f} (± {scores.std():.3f})")

In [ ]:
# 3) Final check of the chosen model on the untouched test set
final_model = Pipeline([("prep", preprocess), ("model", models["Logistic Regression"])])
final_model.fit(X_train, y_train)

y_true = y_test
y_pred_proba = final_model.predict_proba(X_test)[:, 1]

# 4) The decision threshold reflects the precision–recall trade-off for the business
threshold = 0.5
y_pred = (y_pred_proba >= threshold).astype(int)

print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
print("Precision:", round(precision_score(y_true, y_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_true, y_pred, zero_division=0), 4))
print("F1-score:", round(f1_score(y_true, y_pred, zero_division=0), 4))
print("ROC-AUC:", round(roc_auc_score(y_true, y_pred_proba), 4))
print("PR-AUC (Average Precision):", round(average_precision_score(y_true, y_pred_proba), 4))
print("\nConfusion matrix:\n", confusion_matrix(y_true, y_pred))
print("\nClassification report:\n", classification_report(y_true, y_pred, zero_division=0))

### Your answer
**Model choice.** I would start with **Logistic Regression** as the main model, because loan decisions must be explainable to customers and regulators, and logistic regression gives clear coefficients that show how each feature raises or lowers the risk, while also giving well-calibrated probabilities. I would compare it with a **Random Forest** (or a gradient-boosting model such as XGBoost), because tree-based models capture non-linear relationships and interactions – for example, that a low income matters more when the loan amount is high – and they are robust to outliers and skewed incomes. If the more complex model is clearly better, explanation tools like feature importance or SHAP values can be used to keep its decisions understandable.

**Evaluation plan.** I would first split the data into a **training set (80%) and a test set (20%) with stratification**, so both keep the same share of risky applications, and I would not touch the test set until the final evaluation. On the training set, I would use **stratified 5-fold cross-validation** to compare models and tune them. I would report **precision, recall, F1-score, ROC-AUC and PR-AUC**, plus the **confusion matrix**; accuracy alone is misleading here, because a model that approves everybody would already be right about 69% of the time. **Recall** on the risky class is especially important, because missing a risky applicant (a future default) usually costs the bank much more than rejecting a good one. I would choose the **decision threshold** by plotting precision and recall across thresholds and picking the point that minimises the expected cost of errors for the bank, rather than automatically using 0.5.

**What the demonstration above shows.** Both models reach a cross-validated ROC-AUC of about **0.77**, so the more complex Random Forest brings no real gain here, and I keep the simpler, more explainable Logistic Regression. On the test set it reaches about 76% accuracy, but its **recall for risky applicants is only ~53%**, meaning it misses almost half of them at the default 0.5 threshold. Lowering the threshold would catch more risky applicants at the cost of rejecting more good ones, which is exactly the business trade-off described above.

**Class imbalance.** Only about 31% of applications are in the risky class, so I would use **stratified splits and folds** to keep this proportion everywhere, use **`class_weight="balanced"`** so that mistakes on the smaller class count more during training, and, if needed, try **resampling** (such as SMOTE or undersampling) applied **only to the training data** inside each fold.

**Hyperparameter tuning.** I would tune hyperparameters (e.g. regularisation strength C for logistic regression; number of trees, max depth and minimum samples per leaf for the random forest) with **GridSearchCV or RandomizedSearchCV** on the training set only. All preprocessing (imputation, scaling, encoding, resampling) would be placed inside a **scikit-learn Pipeline**, so it is re-fitted within each fold and no information from validation or test data leaks into training.

## 🌟 Exercise 4: Designing Machine Learning Solutions for Specific Problems
### Instructions
For each of these scenarios, decide which type of machine learning would be most suitable. Explain.
- Predicting Stock Prices: predict future prices
- Organizing a Library of Books: group books into genres or categories based on similarities
- Program a robot to navigate and find the shortest path in a maze

### Your answer

**1. Predicting stock prices → Supervised learning (regression, on time-series data).**
- *Input data:* historical data for the stock, such as past daily prices, trading volume, moving averages, and possibly market indicators or news sentiment, arranged in time order.
- *Output:* a **number** – the predicted price (or price change) for the next day or period.
- *Learning signal / objective:* the actual future prices in the historical data act as labels, and the model learns by minimising the error between its predictions and those real values (e.g. mean squared error).

It's supervised because every training example has a known correct answer (what the price really was), and it's regression because that answer is a continuous number. (Note: stock prices are very noisy, so even good models predict them only weakly.)

**2. Organizing a library of books into genres → Unsupervised learning (clustering).**
- *Input data:* descriptions of each book, such as the text, summary, keywords, author, or reader behaviour, turned into numerical features (e.g. word frequencies or text embeddings).
- *Output:* a **group (cluster) number** for each book, so that similar books end up together.
- *Learning signal / objective:* there are no labels; the algorithm (e.g. K-means or hierarchical clustering) tries to make books inside a cluster as similar as possible and clusters as different as possible from each other.

It's unsupervised because we don't give the algorithm the "right" genre – it discovers the groups by itself, and a person then looks at each cluster and names it.

**3. Programming a robot to find the shortest path in a maze → Reinforcement learning.**
- *Input data:* the robot's current **state** – its position in the maze and what it can sense around it (walls, open paths).
- *Output:* an **action** at each step (move up, down, left or right), which together form a path through the maze.
- *Learning signal / objective:* **rewards** – for example a big positive reward for reaching the exit and a small penalty for every step or for hitting a wall – and the agent learns a policy that maximises the total reward, which means reaching the exit in as few steps as possible.

It's reinforcement learning because there is no dataset of correct answers; the robot learns by trial and error, from the consequences of its own actions.

## 🌟 Exercise 5: Designing an Evaluation Strategy for Different ML Models
### Instructions
Select three types of models (supervised, unsupervised, reinforcement learning) and describe how you would evaluate each, including the challenges and limitations.

In [ ]:
# Supervised classification metrics – this reuses the real predictions from Exercise 3
print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
print("Precision:", round(precision_score(y_true, y_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_true, y_pred, zero_division=0), 4))
print("F1-score:", round(f1_score(y_true, y_pred, zero_division=0), 4))
print("ROC-AUC:", round(roc_auc_score(y_true, y_pred_proba), 4))
print("PR-AUC (Average Precision):", round(average_precision_score(y_true, y_pred_proba), 4))

In [ ]:
# Unsupervised clustering: elbow method + silhouette score on synthetic data
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

X_blobs, _ = make_blobs(n_samples=300, centers=3, random_state=42)

ks = range(2, 9)
inertias, silhouettes = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init="auto", random_state=42)
    labels = km.fit_predict(X_blobs)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_blobs, labels))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(ks, inertias, marker="o")
axes[0].set_title("Elbow method (inertia)")
axes[0].set_xlabel("k")
axes[1].plot(ks, silhouettes, marker="o", color="seagreen")
axes[1].set_title("Silhouette score (higher is better)")
axes[1].set_xlabel("k")
plt.show()

print("Best k by silhouette:", list(ks)[int(np.argmax(silhouettes))])

### Your answer

**Supervised model – Random Forest classifier (e.g. loan default prediction).**
First, I would split the data into a **training set and a held-out test set** (e.g. 80/20), using a **stratified** split so both sets keep the same proportion of defaults. On the training set, I would use **5-fold stratified cross-validation** to compare models and tune hyperparameters, and I would touch the test set only once, at the very end, for an honest final score. I would report **accuracy**, but I would focus on **precision** (of the loans we flag as risky, how many really default), **recall** (of all real defaults, how many we catch) and the **F1-score** that balances them, because accuracy is misleading when one class is rare. I would also plot the **ROC curve** and report **ROC-AUC**, plus the **precision-recall curve** (PR-AUC), which is more informative under class imbalance, and I would inspect the **confusion matrix** to see the types of errors. Finally, I would choose the decision threshold based on the business cost of each error (a missed default usually costs more than a rejected good customer). *Key challenge:* when classes are imbalanced or the data changes over time, a high score on past data does not guarantee good performance on future applicants.

**Unsupervised model – K-means clustering (e.g. customer segmentation).**
Because there are no labels, I would evaluate the **quality of the structure** the model finds. I would use the **elbow method** – plotting the within-cluster sum of squares (inertia) for different numbers of clusters *k* and choosing the *k* where the curve bends and adding clusters stops helping much. I would compute the **silhouette score** for each *k* (from −1 to 1; higher means points are close to their own cluster and far from others) and look at a **silhouette plot** for clusters that are weak. Other internal metrics, like the **Davies-Bouldin index** (lower is better) or the **Calinski-Harabasz score** (higher is better), give a second opinion. I would also check that the clusters are **stable** (similar when re-running with different random starts or on a subsample), visualise them in 2D with **PCA**, and – most importantly – check with domain experts whether each cluster makes practical sense. *Key challenge:* there is no "correct answer" to compare against, so a mathematically good clustering is not necessarily a useful one.

**Reinforcement learning model – Q-learning agent (e.g. the maze robot).**
I would track the **cumulative reward per episode** and plot a **learning curve** (often a moving average) across training episodes; a rising curve that levels off shows the agent is learning. I would measure **convergence** by checking that the reward curve and the learned values (Q-values) stop changing much and that the agent's chosen path becomes stable, and I would compare its path length with the true shortest path. To manage the **exploration vs. exploitation** balance, I would use an ε-greedy strategy that starts with a high ε (lots of random exploration) and gradually decreases it, and I would evaluate the final policy separately with exploration turned off. I would also run training with several random seeds, because RL results can vary a lot between runs, and test the agent on new mazes to check that it generalises. *Key challenge:* RL evaluation is noisy and slow, and good performance in the training environment may not transfer to a different or real-world environment.